# 03c2 - Ablasi rumus fusi RAC (RM-c)

RM-c produksi memakai fusi linear-konveks tetap `p_final = (1-alpha)*p_bert +
alpha*p_retrieval` (`RACClassifier.fuse`), dengan titik operasi juara hasil tuning
`alpha=0,2, k=5, weighting=similarity` (val F1-macro 0,9699).

Notebook ini adalah studi ABLASI, bukan perubahan pipeline produksi: menguji empat
rumus fusi dari literatur di atas retrieval (k=5, weighting=similarity) dan head RM-b
yang SAMA, supaya perbandingan mengisolasi kontribusi mekanisme fusi itu sendiri,
bukan hyperparameter retrieval.

1. **Rumus 1** (Long dkk., 2022) - fusi level skor, L2-normalize logit mentah dan
   vote retrieval mentah lalu dirata-ratakan.
2. **Rumus 2** - alpha adaptif = rata-rata similarity k tetangga (parameter-free).
3. **Rumus 3** - alpha adaptif = 1 - keyakinan head (parameter-free).
4. **Rumus 4** - geometric pooling / product-of-experts, alpha di-sweep pada grid
   kecil (0,1 s.d. 0,5).

Split test tetap tertutup; evaluasi hanya di split validation, seperti seluruh
kampanye tuning RM-c lainnya. Hasil ditulis ke folder terpisah
(`outputs/tuning/fusion_ablation/`) dan tidak pernah menyentuh `runs_rmc.csv`,
`best.json`, atau `checkpoints/rmc_best.pt`.

In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

runner = CampaignRunner(out_dir=settings.default_out_dir)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

## 1. Head RM-b terbaik dan fitur beku

In [ ]:
head, head_config = runner._load_best_head()
print("konfigurasi head RM-b terbaik:", head_config)
print("indeks FAISS akan dibangun dari", len(runner.features.labels["train"]), "vektor train")

Indeks dibangun eksklusif dari split train, dan retrieval (k=5, weighting=similarity)
ditahan tetap lintas keempat rumus di bawah - hanya mekanisme fusi yang berubah.

## 2. Jalankan keempat rumus

In [ ]:
from src.services.fusion_ablation import FusionFormulaComparator

comparator = FusionFormulaComparator(
    runner.features, head, runner.device, k=5, weighting="similarity"
)
results = comparator.run_all(split="val", rumus4_alphas=(0.1, 0.2, 0.3, 0.4, 0.5))
results[["formula", "alpha", "val_f1_macro", "val_f1_judi", "val_accuracy", "eval_time_s"]]

## 3. Bandingkan terhadap baseline produksi

In [ ]:
# Referensi tetap: juara RM-c produksi (alpha=0,2, k=5, weighting=similarity),
# dicatat sebagai konstanta, bukan diturunkan ulang dari runs_rmc.csv, supaya
# notebook ablasi ini tidak coupling ke riwayat kampanye.
BASELINE_VAL_F1_MACRO = 0.9699954201283221

ranked = results.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
ranked["delta_vs_baseline_pp"] = (ranked["val_f1_macro"] - BASELINE_VAL_F1_MACRO) * 100
ranked[["formula", "alpha", "val_f1_macro", "val_f1_judi", "delta_vs_baseline_pp"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))

single_formula = ranked[ranked["formula"] != "rumus4"]
ax.bar(single_formula["formula"], single_formula["val_f1_macro"], label="Rumus 1-3")

rumus4 = results[results["formula"] == "rumus4"].sort_values("alpha")
ax.plot(
    ["rumus4"] * len(rumus4),
    rumus4["val_f1_macro"],
    "o",
    color="tab:orange",
    label="Rumus 4 (per alpha)",
)
for _, row in rumus4.iterrows():
    ax.annotate(f"a={row['alpha']:.1f}", (
        "rumus4", row["val_f1_macro"]
    ), textcoords="offset points", xytext=(6, 0), fontsize=8)

ax.axhline(BASELINE_VAL_F1_MACRO, color="gray", linestyle="--", label="Baseline produksi (alpha=0,2, k=5)")
ax.set_ylabel("F1-macro (validation)")
ax.set_title("Perbandingan rumus fusi RAC")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

## 4. Simpan hasil

In [ ]:
from src.utils.io import write_csv

out_path = settings.default_out_dir / "fusion_ablation" / "fusion_ablation_results.csv"
write_csv(out_path, ranked)
print("hasil ditulis ke", out_path)

## Ringkasan

Isi setelah menjalankan notebook ini: rumus mana yang menyamai, mengalahkan, atau
kalah dari fusi linear-konveks produksi (alpha=0,2, k=5), dan apakah selisihnya
berada dalam ambang seri 0,15 pp yang dipakai kampanye tuning RM-c. Kesimpulan ini
untuk Bab 4 - dampaknya pada pipeline produksi RM-c adalah NOL karena
`RACClassifier.fuse` tidak diubah oleh notebook ini.